# TermoRed S.A. — Capa Gold

Objetivo: dejar una tabla lista para que un analista responda qué bancos
cumplieron la cadena de frío y, cuando no, cuál fue la causa probable.

Dataset elegido: banco + día.

Atribución de causa (por excursión):
1. Si la lectura cae dentro de una ventana de corte de energía en la zona
   eléctrica del banco → `corte_energia`.
2. Si no, y la puerta estaba abierta → `puerta_abierta`.
3. Si no aplica ninguna de las anteriores → `sin_causa_identificada`.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CUENTA     = "destorageintegration"
CONTENEDOR = "termored-central-us-dev"
CLAVE = dbutils.secrets.get("termored_scope", "storage_key")

spark.conf.set(f"fs.azure.account.key.{CUENTA}.dfs.core.windows.net", CLAVE)

BASE   = f"abfss://{CONTENEDOR}@{CUENTA}.dfs.core.windows.net"
SILVER = f"{BASE}/2_silver"
GOLD   = f"{BASE}/3_gold"


def guardar_parquet(df, destino, nombre, versionar=False):
    temp_path = f"{destino}_tmp"
    (df.coalesce(1).write.mode("overwrite")
       .option("compression", "snappy").parquet(temp_path))
    archivo = [f.path for f in dbutils.fs.ls(temp_path) if f.name.endswith(".parquet")][0]
    sufijo = "_" + datetime.now().strftime("%Y%m%d_%H%M%S") if versionar else ""
    final = f"{destino}/{nombre}{sufijo}.parquet"
    dbutils.fs.mv(archivo, final)
    dbutils.fs.rm(temp_path, recurse=True)
    return final

## 1. Lectura de Silver

In [0]:
fact_lecturas  = spark.read.parquet(f"{SILVER}/fact_lecturas")
dim_bancos     = spark.read.parquet(f"{SILVER}/dim_bancos")
cortes_energia = spark.read.parquet(f"{SILVER}/cortes_energia")

## 2. Atribución de causa a cada excursión

Se cruza cada lectura fuera de rango con los cortes de energía de la misma
zona eléctrica, verificando que el timestamp caiga dentro de la ventana del
corte.

In [0]:
excursiones = (fact_lecturas
               .filter("fuera_de_rango")
               .join(dim_bancos.select("banco_id", "zona_electrica", "nombre", "ciudad"),
                     on="banco_id", how="left"))

cortes = cortes_energia.select(
    "zona_electrica",
    F.col("fecha_hora_inicio").alias("corte_inicio"),
    F.col("fecha_hora_fin").alias("corte_fin"))

excursiones = excursiones.join(cortes, on="zona_electrica", how="left")

excursiones = (excursiones
    .withColumn("coincide_corte",
                F.col("ts").between(F.col("corte_inicio"), F.col("corte_fin")))
    .groupBy("lectura_id", "banco_id", "nombre", "ciudad", "fecha", "turno",
             "sensor_id", "desvio_c", "puerta_abierta")
    .agg(F.max("coincide_corte").alias("hubo_corte"))
    .withColumn("causa_probable",
                F.when(F.col("hubo_corte"), "corte_energia")
                 .when(F.col("puerta_abierta"), "puerta_abierta")
                 .otherwise("sin_causa_identificada"))
)

display(excursiones.limit(20))

lectura_id,banco_id,nombre,ciudad,fecha,turno,sensor_id,desvio_c,puerta_abierta,hubo_corte,causa_probable
LEC-01709,BSG-007,Banco de Sangre Villa María,Villa María,2025-06-08,Mañana,SEN-0033,2.5,true,false,puerta_abierta
LEC-01264,BSG-009,Banco de Sangre Santa Fe,Santa Fe,2025-06-06,Tarde,SEN-0042,1.2,true,false,puerta_abierta
LEC-01494,BSG-006,Banco de Sangre Río Cuarto,Río Cuarto,2025-06-07,Tarde,SEN-0026,1.9,true,false,puerta_abierta
LEC-00163,BSG-007,Banco de Sangre Villa María,Villa María,2025-06-02,Tarde,SEN-0031,1.5,false,false,sin_causa_identificada
LEC-01661,BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,2025-06-07,Noche,SEN-0005,1.7,true,false,puerta_abierta
LEC-00271,BSG-009,Banco de Sangre Santa Fe,Santa Fe,2025-06-02,Noche,SEN-0041,1.2,true,false,puerta_abierta
LEC-00088,BSG-010,Banco de Sangre Rafaela,Rafaela,2025-06-02,Mañana,SEN-0046,4.4,false,true,corte_energia
LEC-01574,BSG-013,Banco de Sangre San Miguel de Tucumán,San Miguel de Tucumán,2025-06-07,Noche,SEN-0065,2.7,true,false,puerta_abierta
LEC-01130,BSG-012,null,null,2025-06-06,Mañana,SEN-0060,2.5,false,null,sin_causa_identificada
LEC-01565,BSG-016,Banco de Sangre Orán,Orán,2025-06-07,Tarde,SEN-0079,1.8,false,false,sin_causa_identificada


## 3. Agregación: cumplimiento por banco y día

In [0]:
total_lecturas = (fact_lecturas
    .groupBy("banco_id", "fecha")
    .agg(F.count("*").alias("total_lecturas")))

resumen_excursiones = (excursiones
    .groupBy("banco_id", "fecha")
    .agg(
        F.count("*").alias("excursiones"),
        F.round(F.avg("desvio_c"), 2).alias("desvio_promedio_c"),
        F.round(F.max("desvio_c"), 2).alias("desvio_maximo_c"),
        F.sum(F.when(F.col("causa_probable") == "corte_energia", 1).otherwise(0)).alias("excursiones_por_corte"),
        F.sum(F.when(F.col("causa_probable") == "puerta_abierta", 1).otherwise(0)).alias("excursiones_por_puerta"),
        F.sum(F.when(F.col("causa_probable") == "sin_causa_identificada", 1).otherwise(0)).alias("excursiones_sin_causa"),
        F.sum(F.when((F.col("causa_probable") == "puerta_abierta") & (F.col("turno") == "Noche"), 1).otherwise(0)).alias("excursiones_puerta_turno_noche"),
    ))

gold_cumplimiento = (total_lecturas
    .join(resumen_excursiones, on=["banco_id", "fecha"], how="left")
    .join(dim_bancos.select("banco_id", "nombre", "ciudad", "provincia"), on="banco_id", how="inner")
    .fillna(0, subset=["excursiones", "excursiones_por_corte", "excursiones_por_puerta",
                        "excursiones_sin_causa", "excursiones_puerta_turno_noche"])
    .withColumn("tasa_cumplimiento_pct",
                F.round(100 * (1 - F.col("excursiones") / F.col("total_lecturas")), 2))
    .withColumn("cumple_cadena_frio", F.col("excursiones") == 0)
    .select("banco_id", "nombre", "ciudad", "provincia", "fecha",
            "total_lecturas", "excursiones", "tasa_cumplimiento_pct", "cumple_cadena_frio",
            "desvio_promedio_c", "desvio_maximo_c",
            "excursiones_por_corte", "excursiones_por_puerta",
            "excursiones_puerta_turno_noche", "excursiones_sin_causa")
    .orderBy("banco_id", "fecha")
)

display(gold_cumplimiento)

banco_id,nombre,ciudad,provincia,fecha,total_lecturas,excursiones,tasa_cumplimiento_pct,cumple_cadena_frio,desvio_promedio_c,desvio_maximo_c,excursiones_por_corte,excursiones_por_puerta,excursiones_puerta_turno_noche,excursiones_sin_causa
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-02,13,1,92.31,false,3.3,3.3,0,1,1,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-03,13,5,61.54,false,4.66,5.8,5,0,0,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-04,11,0,100.0,true,null,null,0,0,0,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-05,14,5,64.29,false,4.26,6.0,5,0,0,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-06,11,0,100.0,true,null,null,0,0,0,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-07,14,6,57.14,false,3.53,4.7,5,1,1,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-08,13,0,100.0,true,null,null,0,0,0,0
BSG-002,Banco de Sangre La Plata,La Plata,Buenos Aires,2025-06-02,11,0,100.0,true,null,null,0,0,0,0
BSG-002,Banco de Sangre La Plata,La Plata,Buenos Aires,2025-06-03,14,9,35.71,false,3.6,5.8,9,0,0,0
BSG-002,Banco de Sangre La Plata,La Plata,Buenos Aires,2025-06-04,13,0,100.0,true,null,null,0,0,0,0


## 4. Persistencia en Gold

In [0]:
ruta = guardar_parquet(gold_cumplimiento, f"{GOLD}/cumplimiento_cadena_frio", "cumplimiento_cadena_frio")
print(f"OK  cumplimiento_cadena_frio  {gold_cumplimiento.count()} filas -> {ruta}")

OK  cumplimiento_cadena_frio  133 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/3_gold/cumplimiento_cadena_frio/cumplimiento_cadena_frio.parquet


## 5. Verificación

In [0]:
leido = spark.read.parquet(f"{GOLD}/cumplimiento_cadena_frio")
print(f"{leido.count()} filas | {len(leido.columns)} columnas")
display(leido.orderBy("tasa_cumplimiento_pct").limit(20))

133 filas | 15 columnas


banco_id,nombre,ciudad,provincia,fecha,total_lecturas,excursiones,tasa_cumplimiento_pct,cumple_cadena_frio,desvio_promedio_c,desvio_maximo_c,excursiones_por_corte,excursiones_por_puerta,excursiones_puerta_turno_noche,excursiones_sin_causa
BSG-008,Banco de Sangre Rosario,Rosario,Santa Fe,2025-06-03,13,10,23.08,false,3.58,5.6,10,0,0,0
BSG-002,Banco de Sangre La Plata,La Plata,Buenos Aires,2025-06-03,14,9,35.71,false,3.6,5.8,9,0,0,0
BSG-014,Banco de Sangre Concepción,Concepción,Tucuman,2025-06-03,15,9,40.0,false,3.17,4.7,7,1,0,1
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-07,14,6,57.14,false,3.53,4.7,5,1,1,0
BSG-010,Banco de Sangre Rafaela,Rafaela,Santa Fe,2025-06-04,10,4,60.0,false,4.35,5.7,4,0,0,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-03,13,5,61.54,false,4.66,5.8,5,0,0,0
BSG-004,Banco de Sangre Bahía Blanca,Bahía Blanca,Buenos Aires,2025-06-04,14,5,64.29,false,3.46,6.0,5,0,0,0
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,2025-06-05,14,5,64.29,false,4.26,6.0,5,0,0,0
BSG-018,Banco de Sangre Paraná,Paraná,Entre Rios,2025-06-06,14,5,64.29,false,2.4,3.5,0,1,1,4
BSG-004,Banco de Sangre Bahía Blanca,Bahía Blanca,Buenos Aires,2025-06-02,14,5,64.29,false,4.1,6.0,5,0,0,0


In [0]:
gold_cumplimiento.filter(F.col("nombre").isNull()).show()

+--------+------+------+---------+-----+--------------+-----------+---------------------+------------------+-----------------+---------------+---------------------+----------------------+------------------------------+---------------------+
|banco_id|nombre|ciudad|provincia|fecha|total_lecturas|excursiones|tasa_cumplimiento_pct|cumple_cadena_frio|desvio_promedio_c|desvio_maximo_c|excursiones_por_corte|excursiones_por_puerta|excursiones_puerta_turno_noche|excursiones_sin_causa|
+--------+------+------+---------+-----+--------------+-----------+---------------------+------------------+-----------------+---------------+---------------------+----------------------+------------------------------+---------------------+
+--------+------+------+---------+-----+--------------+-----------+---------------------+------------------+-----------------+---------------+---------------------+----------------------+------------------------------+---------------------+



In [0]:
tasa_por_banco = (gold_cumplimiento.groupBy('banco_id', 'nombre').agg(F.round(F.avg('tasa_cumplimiento_pct'), 2).alias('tasa_cumplimiento_promedio')).orderBy('tasa_cumplimiento_promedio')) 

display(tasa_por_banco)

banco_id,nombre,tasa_cumplimiento_promedio
BSG-001,Banco de Sangre Ciudad de Buenos Aires,82.18
BSG-004,Banco de Sangre Bahía Blanca,87.89
BSG-010,Banco de Sangre Rafaela,88.57
BSG-008,Banco de Sangre Rosario,89.01
BSG-002,Banco de Sangre La Plata,89.8
BSG-018,Banco de Sangre Paraná,89.9
BSG-014,Banco de Sangre Concepción,90.0
BSG-007,Banco de Sangre Villa María,90.1
BSG-020,Banco de Sangre Neuquén,90.14
BSG-011,Banco de Sangre Mendoza,91.5


Databricks visualization. Run in Databricks to view.